<a href="https://colab.research.google.com/github/Blackcyan30/cs_421_project_sp26/blob/main/CS421_Project1_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# 1. Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 0.001
EPOCHS = 10

# 2. Data Preprocessing & Tokenization
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def build_vocab(texts, max_vocab=10000):
    all_words = []
    for t in texts:
        all_words.extend(tokenize(t))
    counts = Counter(all_words)
    vocab = {word: i+2 for i, (word, _) in enumerate(counts.most_common(max_vocab))}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

def text_to_indices(text, vocab, max_len=50):
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return torch.tensor(indices), len(tokens)

# FIXED: Dataset now accepts a DataFrame directly to avoid repeated parsing errors
class WASSADataset(Dataset):
    def __init__(self, df, vocab, is_test=False):
        self.df = df
        self.vocab = vocab
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text_indices, length = text_to_indices(row['text'], self.vocab)

        if self.is_test:
            return str(row['id']), text_indices, length

        # Polarity mapping: Adjusting to ensure integers are used for CrossEntropy
        polarity_map = {-1: 0, 0: 1, 1: 2}

        return {
            'text': text_indices,
            'length': length,
            'emotion': torch.tensor(float(row['Emotion']), dtype=torch.float),
            'empathy': torch.tensor(float(row['Empathy']), dtype=torch.float),
            'polarity': torch.tensor(polarity_map.get(int(row['EmotionalPolarity']), 1), dtype=torch.long)
        }

# 3. Multi-Task RNN Model
class MultiTaskRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

        self.emotion_head = nn.Linear(hid_dim, 1)        # Regression
        self.empathy_head = nn.Linear(hid_dim, 1)        # Regression
        self.polarity_head = nn.Linear(hid_dim, 3)       # Classification

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        _, (hidden, _) = self.rnn(embedded)
        last_hidden = self.dropout(hidden[-1])

        return (self.emotion_head(last_hidden).squeeze(),
                self.empathy_head(last_hidden).squeeze(),
                self.polarity_head(last_hidden))

# 4. FIXED: Robust Loading Section
def safe_load_csv(file_path):
    # Using 'python' engine and on_bad_lines='skip' to bypass the formatting errors in the CSV
    return pd.read_csv(
        file_path,
        engine='python',
        on_bad_lines='skip',
        quoting=0,
        escapechar='\\'
    )

print("Loading Data...")
train_df = safe_load_csv('trac2_CONVT_train.csv')
dev_df = safe_load_csv('trac2_CONVT_dev.csv')
test_df = safe_load_csv('trac2_CONVT_test.csv')

vocab = build_vocab(train_df['text'].tolist())

# Initializing datasets with DataFrames
train_ds = WASSADataset(train_df, vocab)
dev_ds = WASSADataset(dev_df, vocab)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE)

model = MultiTaskRNN(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
mse_loss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()

print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        txt = batch['text'].to(DEVICE)
        emo_p, emp_p, pol_p = model(txt)

        loss_emo = mse_loss(emo_p, batch['emotion'].to(DEVICE))
        loss_emp = mse_loss(emp_p, batch['empathy'].to(DEVICE))
        loss_pol = ce_loss(pol_p, batch['polarity'].to(DEVICE))

        loss = loss_emo + loss_emp + loss_pol
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

# 5. Generate Test Predictions
print("Generating Predictions...")
test_ds = WASSADataset(test_df, vocab, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
model.eval()

results = []
rev_polarity_map = {0: -1, 1: 0, 2: 1}

with torch.no_grad():
    for ids, txt, lengths in test_loader:
        emo_p, emp_p, pol_p = model(txt.to(DEVICE))
        pol_labels = torch.argmax(pol_p, dim=1).cpu().numpy()

        # Handle potential single-item batches
        emo_p = emo_p.cpu().view(-1)
        emp_p = emp_p.cpu().view(-1)

        for i in range(len(ids)):
            results.append({
                'id': ids[i],
                'Emotion': emo_p[i].item(),
                'EmotionalPolarity': rev_polarity_map[pol_labels[i]],
                'Empathy': emp_p[i].item()
            })

pd.DataFrame(results).to_csv('predictions_rnn.csv', index=False)
print("Saved predictions_rnn.csv")

Loading Data...
Starting Training...
Epoch 1/10, Loss: 2.4037
Epoch 2/10, Loss: 2.0468
Epoch 3/10, Loss: 1.9040
Epoch 4/10, Loss: 1.7727
Epoch 5/10, Loss: 1.6886
Epoch 6/10, Loss: 1.6347
Epoch 7/10, Loss: 1.5790
Epoch 8/10, Loss: 1.5372
Epoch 9/10, Loss: 1.4943
Epoch 10/10, Loss: 1.4758
Generating Predictions...
Saved predictions_rnn.csv
